In [2]:
from pathlib import Path
import pandas as pd 
import glob
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()
import xgboost as xgb
from scipy.optimize import minimize_scalar
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from scipy.stats import pearsonr, spearmanr

root = Path('/data/data/malpolon/xgb/')
inputs_path = Path('/marbec-data/RLS-Australia/malpolon/inputs/australia/')
output_path = Path('/marbec-data/RLS-Australia/malpolon/outputs/')

In [3]:
fulldf = pd.read_csv(inputs_path / 'database_common.csv', index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
species = list(fulldf.columns[-818:-1])
groundtruth = fulldf[species]

In [20]:
## Bins data
groundtruth_bins = {}
for i in [5, 10, 20]:
    df = pd.read_csv(inputs_path / f"database_common_{i}binned.csv", index_col='survey_id',
                     dtype = {22:str, 24:str, 25:str})
    groundtruth_bins[i] = df[species]

## Preparation

In [6]:
## Calculate predictors
 
def get_compound_values(row):

    survey_id = row.name

    filename = Path(inputs_path) / "env" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    env = np.transpose(x, (2, 0, 1))

    filename = Path(inputs_path) / "hum" / (str(survey_id) + '.npy')
    x = np.load(filename).astype(np.float32)
    hum = np.transpose(x, (2, 0, 1))

    envhum = np.concatenate([env, hum], axis=0)

    center_values = 0.25 * (envhum[:, 16, 16] + envhum[:, 15, 16] + envhum[:, 16, 15] + envhum[:, 15, 15])
    means = np.mean(envhum, axis=(1,2))
    deviations = np.std(envhum, axis=(1,2)) / means


    filename = Path(inputs_path) / "dhw" / (str(survey_id) + '.npy')
    dhw = np.load(filename).astype(np.float32)
    dhw_vect = np.array([np.mean(dhw), np.std(dhw) / np.mean(dhw), np.max(dhw)])

    return np.concatenate([center_values, means, deviations, dhw_vect])

# X = fulldf[['subset']].progress_apply(get_compound_values, axis=1, result_type='expand')
# X.to_csv(root / f'X_compound_values_mm.csv')

In [ ]:
## Prepare datasets

X = pd.read_csv(root / f'X_compound_values_mm.csv', index_col = 0)

def get_datasets(sp, modeltype='rf', objective='pa', num_bins=None):

    # X
    X_train = X.loc[fulldf['subset'] == 'train']
    X_val = X.loc[fulldf['subset'] == 'val']
    X_test = X.loc[fulldf['subset'] == 'test']

    # Y
    if objective == 'pa':
        targets = (groundtruth[sp] > 0).astype("category")
    elif objective == 'bins':
        targets = groundtruth_bins[num_bins][sp].astype("category")
    else:
        targets = np.log(1+groundtruth[sp].astype(float))

    Y_train = targets.loc[fulldf['subset'] == 'train']
    Y_val = targets.loc[fulldf['subset'] == 'val']
    Y_test = targets.loc[fulldf['subset'] == 'test']

    if modeltype == 'xgb':

        dtrain_clf = xgb.DMatrix(X_train, Y_train, enable_categorical=True)
        dval_clf = xgb.DMatrix(X_val, Y_val, enable_categorical=True)
        dtest_clf = xgb.DMatrix(X_test, Y_test, enable_categorical=True)

        return dtrain_clf, dval_clf, dtest_clf
    
    else:

        return X_train, Y_train, X_val, Y_val, X_test, Y_test

## P-A

#### Train XGB

In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
   for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'pa')
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "binary:logistic", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4,"eval_metric":"auc","base_score":0.5}
      n=500

      model = xgb.train(
         params=params,
         dtrain=dtrain_clf,
         num_boost_round=n,
         evals=evals,
         verbose_eval=None,
         #early_stopping_rounds=10
      )

      targets = (groundtruth[sp] > 0).astype(float)
      Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
      Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
      Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

#### Binarization and val F1

In [6]:
def rvalue(thres, df, target_sr):
    sr = (df > thres).astype(int).sum(axis=1)
    
    return np.abs(sr.mean() - target_sr.mean())


for ss in np.arange(0.1, 1.0, 0.2):

    # Calculate threshold

    xgbdict = {}

    for sp in tqdm(species):

        xgbdict[sp] = pd.read_csv(root / 'pa' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()

    xgb_val = pd.DataFrame(xgbdict)

    targets_sr = (fulldf.loc[xgb_val.index, xgb_val.columns] > 0).sum(axis=1)
    xgb_threshold = minimize_scalar(rvalue, args=(xgb_val, targets_sr),method='Bounded', bounds=(0,1))['x']

    # Calculate val F1-scores

    xgb_val_pa = (pd.DataFrame(xgbdict) > xgb_threshold)

    for s in species:

        targ = (fulldf.loc[xgb_val_pa.index, s] > 0).astype(int).to_numpy().flatten()
        xgb_preds = xgb_val_pa[s].astype(int).to_numpy().flatten()
        xgbdict[s] = {'f1': f1_score(targ, xgb_preds, zero_division=0)}


    xgb_val_f1 = pd.DataFrame(xgbdict).T.sort_values('f1', ascending=False)
    xgb_val_f1.to_csv(root / 'pa' / f"xgb_val_f1-{ss:.1f}-{xgb_threshold:.3f}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate F1 on test set

In [7]:
def best_f1(row):
    sp = row.name
    ss = row['ss']
    threshold = row['threshold']

    preds_test = pd.read_csv(root / 'pa' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    preds_test_pa = (preds_test > threshold).astype(int).to_numpy().flatten()

    targ = (fulldf.loc[preds_test.index, sp] > 0).astype(int).to_numpy().flatten()

    return(f1_score(targ, preds_test_pa, zero_division=0))


In [8]:
scoresxgb = pd.DataFrame(index = species)
threshold = {}

for ss in np.arange(0.1, 1.0, 0.2):

    p = next(Path(root / 'pa').glob(f"xgb_val_f1-{ss:.1f}-*.csv"))
    scoresxgb[f"{ss:.1f}"] = pd.read_csv(p, index_col = 0)['f1']
    threshold[f"{ss:.1f}"] = float(p.stem.split('-')[-1])

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1)
best_idx['threshold'] = best_idx['ss'].map(threshold)

f1 = best_idx.progress_apply(best_f1, axis=1)
xgb_f1 = pd.DataFrame(f1, columns=["f1"]).sort_values('f1', ascending=False)
xgb_f1.to_csv(root / 'pa' / f"xgbbest_f1--.4rank={len(xgb_f1[xgb_f1['f1']>=0.4])}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

#### Plot comparison chart

In [15]:
xgbbest_f1 = pd.read_csv(next((root / 'pa').glob("xgbbest_f1*.csv")), index_col = 0)

cp = '26_hum_env_dhw_bathy_common_pa-2025-11-10_16-31'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

In [16]:
scores = pd.concat([xgbbest_f1["f1"], mm_f1["f1"], td_f1["f1"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'F1']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [17]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='F1', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Presence-absence classification',
    xaxis_title='Species rank',
    yaxis_title='F1 score',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

## Bins

#### Train XGB

In [24]:
i = 10
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

        dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'bins', i)
        evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

        params = {"objective": "multi:softmax", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4,"num_class":i}
        n=100

        model = xgb.train(
            params=params,
            dtrain=dtrain_clf,
            num_boost_round=n,
            evals=evals,
            verbose_eval=None
        )

        targets = (groundtruth[sp] > 0).astype(float)
        Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
        Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
        Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

        pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"xgb{i}-preds-{ss:.1f}" / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate R2 on val set

In [ ]:
## val

i = 10
earlystopping = '-es'

for ss in np.arange(0.1, 1.0, 0.2):

    # Calculate threshold

    xgbdict = {}

    for sp in tqdm(species):

        xgb_val = pd.read_csv(root / 'bins' / f'xgb{i}-preds-{ss:.1f}{earlystopping}' / f"val_{sp.replace('/','-')}.csv", index_col=0).squeeze()
        targ = groundtruth_bins[i].loc[xgb_val.index, sp]
        xgbdict[sp] = {'pearsonr': pearsonr(groundtruth_bins[i].loc[xgb_val.index, sp], xgb_val)[0]}


    xgb_val_f1 = pd.DataFrame(xgbdict).T.sort_values('pearsonr', ascending=False)
    xgb_val_f1.to_csv(root / 'bins' / f"xgb{i}_val_r2-{ss:.1f}{earlystopping}.csv")

#### Calculate test R2

In [27]:
i = 10

with open(inputs_path / f"database_common_{i}bins.txt", "r") as f:
        medians = f.readlines()
        medians = [float(m.strip()) for m in medians]

        median_dic = {i: medians[i] for i in range(len(medians))}


def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'bins' / f"xgb{i}-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = groundtruth_bins[i].loc[preds_test.index, sp]

    median_preds_xgb = preds_test.astype(int).replace(median_dic)
    median_pearson = pearsonr(groundtruth.loc[preds_test.index, sp], median_preds_xgb)[0]

    return(pearsonr(targ, preds_test)[0], median_pearson)


In [ ]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'bins' / f"xgb{i}_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')


r2s = best_idx.progress_apply(best_r2, axis=1, result_type='expand')
r2s.columns = ['pearsonr', 'pearsonr_median']
xgb_r2s = r2s.sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'bins' / f"xgb{i}_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")

#### Plot comparison charts

In [ ]:
NBINS = 10

xgbbest_f1 = pd.read_csv(next((root / 'bins').glob(f"xgb{NBINS}_best_r2*19.csv")), index_col = 0)

cp = '39_mm4_transductif_bins-2026-01-22_14-49'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '39_mm4_transductif_bins-2025-12-18_16-50'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

In [39]:
scores = pd.concat([xgbbest_f1["pearsonr"], mm_f1["pearsonr"], td_f1["pearsonr"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'pearsonr']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [40]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='pearsonr', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Biomass classification into 10 bins',
    xaxis_title='Species rank',
    yaxis_title='Pearson R',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

## Reg

#### Train XGB

In [20]:
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp, 'xgb', 'reg')
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4}
      n=500

      model = xgb.train(
        params=params,
        dtrain=dtrain_clf,
        num_boost_round=n,
        evals=evals,
        verbose_eval=None,
        early_stopping_rounds=10
      )

      targets = (groundtruth[sp] > 0).astype(float)
      Y_train = targets.loc[fulldf['subset'] == 'train'].astype('category')
      Y_val = targets.loc[fulldf['subset'] == 'val'].astype('category')
      Y_test = targets.loc[fulldf['subset'] == 'test'].astype('category')

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'reg' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

  0%|          | 0/817 [00:00<?, ?it/s]

#### Calculate R2 on val set

In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
    dic = {}
    for s in tqdm(species):

        predsxgb = pd.read_csv(root / 'reg' / f"xgb-preds-{ss:.1f}" / f"val_{s.replace('/','-')}.csv", index_col=0).squeeze()
        
        dic[s] = {'pearsonr': pearsonr(groundtruth.loc[predsxgb.index, s], np.exp(predsxgb)-1)[0]}
        
    scoresxgb = pd.DataFrame(dic).T.sort_values(ascending=False, by='pearsonr')
    scoresxgb.to_csv(root / 'reg' / f"xgb_val_r2-{ss:.1f}.csv")

#### Calculate test R2

In [22]:
def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'reg' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = groundtruth.loc[preds_test.index, sp]

    return(pearsonr(targ, preds_test)[0])

In [23]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'reg' / f"xgb_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')

r2s = best_idx.progress_apply(best_r2, axis=1)
xgb_r2s = pd.DataFrame(r2s, columns=["pearsonr"]).sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'reg' / f"xgb_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")

/tmp/ipykernel_24863/3918493975.py:8: FutureWarning:

The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError



  0%|          | 0/817 [00:00<?, ?it/s]

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

An input array is constant; the correlation coefficient is not defined.

/tmp/ipykernel_24863/2779842727.py:8: ConstantInputWarning:

A

#### Plot comparison chart

In [4]:
xgbbest_f1 = pd.read_csv(next((root / 'reg').glob("xgb_best_r2*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2026-01-07_16-44'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2025-12-17_15-41'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

In [6]:
scores = pd.concat([xgbbest_f1["pearsonr"], mm_f1["pearsonr"], td_f1["pearsonr"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'pearsonr']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [7]:
import plotly.express as px

fig = px.line(data[data['x'] <= 400], x='x', y='pearsonr', color='model', template = 'simple_white',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Biomass regression',
    xaxis_title='Species rank',
    yaxis_title='Pearson R',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)

fig.update_traces(line={'width': 4})

## Eco indicators

#### Train XGB

In [ ]:
## Prepare datasets
eco_gt = pd.read_csv(inputs_path / 'eco_indic_norm.csv', index_col='survey_id')
species = list(eco_gt.columns)[1:-1]
targets = eco_gt[species]

def get_datasets(sp):

    # X
    X = pd.read_csv(root / f'X_compound_values_mm.csv', index_col = 0)
    X_train = X.loc[eco_gt[eco_gt['subset'] == 'train'].index]
    X_val = X.loc[eco_gt[eco_gt['subset'] == 'val'].index]
    X_test = X.loc[eco_gt[eco_gt['subset'] == 'test'].index]

    # Y
    targets = eco_gt[sp]

    Y_train = targets.loc[eco_gt['subset'] == 'train']
    Y_val = targets.loc[eco_gt['subset'] == 'val']
    Y_test = targets.loc[eco_gt['subset'] == 'test']

    dtrain_clf = xgb.DMatrix(X_train, Y_train, enable_categorical=True)
    dval_clf = xgb.DMatrix(X_val, Y_val, enable_categorical=True)
    dtest_clf = xgb.DMatrix(X_test, Y_test, enable_categorical=True)

    return dtrain_clf, dval_clf, dtest_clf


In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
    for sp in tqdm(species):

      dtrain_clf, dval_clf, dtest_clf = get_datasets(sp)
      evals = [(dtrain_clf, "train"), (dval_clf, "validation")]

      params = {"objective": "reg:squarederror", "tree_method": "hist", "device": "cuda","subsample":ss,"max_depth":4}
      n=200

      model = xgb.train(
        params=params,
        dtrain=dtrain_clf,
        num_boost_round=n,
        evals=evals,
        verbose_eval=None,
        early_stopping_rounds=10
      )

      Y_train = eco_gt.loc[eco_gt['subset'] == 'train']
      Y_val = eco_gt.loc[eco_gt['subset'] == 'val']
      Y_test = eco_gt.loc[eco_gt['subset'] == 'test']

      pd.Series(model.predict(dtrain_clf), name=sp, index = Y_train.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"train_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dval_clf), name=sp, index = Y_val.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"val_{sp.replace('/','-')}.csv")
      pd.Series(model.predict(dtest_clf), name=sp, index = Y_test.index).to_csv(root / 'eco' / f'xgb-preds-{ss:.1f}' / f"test_{sp.replace('/','-')}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

[0]	train-rmse:0.14058	validation-rmse:0.13953
[5]	train-rmse:0.08023	validation-rmse:0.11699
[10]	train-rmse:0.07313	validation-rmse:0.11703
[15]	train-rmse:0.07079	validation-rmse:0.11700
[16]	train-rmse:0.07045	validation-rmse:0.11717
[0]	train-rmse:0.06028	validation-rmse:0.05131
[5]	train-rmse:0.04771	validation-rmse:0.05209
[10]	train-rmse:0.04483	validation-rmse:0.05317
[11]	train-rmse:0.04459	validation-rmse:0.05316
[0]	train-rmse:0.20434	validation-rmse:0.23340
[5]	train-rmse:0.17244	validation-rmse:0.21242
[10]	train-rmse:0.16812	validation-rmse:0.20213
[15]	train-rmse:0.16565	validation-rmse:0.20464
[20]	train-rmse:0.16380	validation-rmse:0.20365


#### Calculate R2 on val set

In [ ]:
for ss in np.arange(0.1, 1.0, 0.2):
    dic = {}
    for s in tqdm(species):

        predsxgb = pd.read_csv(root / 'eco' / f"xgb-preds-{ss:.1f}" / f"val_{s.replace('/','-')}.csv", index_col=0).squeeze()
        
        dic[s] = {'pearsonr': pearsonr(targets.loc[predsxgb.index, s], predsxgb)[0]}
        
    scoresxgb = pd.DataFrame(dic).T.sort_values(ascending=False, by='pearsonr')
    scoresxgb.to_csv(root / 'eco' / f"xgb_val_r2-{ss:.1f}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

#### Calculate test R2

In [ ]:
def best_r2(row):
    sp = row.name
    ss = row['ss']

    preds_test = pd.read_csv(root / 'eco' / f"xgb-preds-{ss}" / f"test_{sp.replace('/','-')}.csv", index_col=0).squeeze()
    targ = eco_gt.loc[preds_test.index, sp]

    return(pearsonr(targ, preds_test)[0])

In [ ]:
scoresxgb = pd.DataFrame(index = species)

for ss in np.arange(0.1, 1.0, 0.2):

    scoresxgb[f"{ss:.1f}"] = pd.read_csv(root / 'eco' / f"xgb_val_r2-{ss:.1f}.csv", index_col = 0)['pearsonr']

best_idx = pd.DataFrame()
best_idx['ss'] = scoresxgb.idxmax(axis=1).fillna('0.1')

r2s = best_idx.progress_apply(best_r2, axis=1)
xgb_r2s = pd.DataFrame(r2s, columns=["pearsonr"]).sort_values('pearsonr', ascending=False).fillna(-.1)
xgb_r2s.to_csv(root / 'eco' / f"xgb_best_r2--.4rank={len(xgb_r2s[xgb_r2s['pearsonr']>=0.4])}.csv")

  0%|          | 0/3 [00:00<?, ?it/s]

#### Plot comparison chart

In [34]:
xgb_f1 = pd.read_csv(next((root / 'eco').glob("xgb_best_r2*.csv")), index_col = 0)

cp = '40_mm4_transductif_eco-2026-01-23_17-45'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '40_mm4_transductif_eco-2026-01-23_16-56'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

tdstacked_f1 = pd.read_csv('~/eco_indic_corr.csv', index_col=0)

In [35]:
scores = pd.concat([xgb_f1["pearsonr"], mm_f1["pearsonr"], td_f1["pearsonr"], tdstacked_f1["pearsonr"]], axis = 1)
names = ['XGBoost', 'Deep-SDM', 'Transductive (direct)', 'Transductive (computed)']
scores.columns = names

y = []

for n in names:
    y1 = scores[[n]].sort_values(ascending=False, by=n)
    y1.reset_index(inplace=True)
    y1.columns = ['species', 'pearsonr']
    y1['x'] = np.arange(len(y1))
    y1['model'] = n
    y.append(y1)

data = pd.concat(y)

In [36]:
import plotly.express as px

fig = px.bar(data, x=data['species'], y='pearsonr', color='model', template = 'simple_white', barmode='group',
           width = 800, height=800,
           color_discrete_sequence = px.colors.qualitative.Pastel)

fig.update_layout(
    title='Ecological indicators regression',
    xaxis_title='',
    yaxis_title='Pearson R',
    font=dict(size=20),
    legend_title_text='',
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)


## Plot trait-based improvement

In [3]:
# P-A

xgb_f1 = pd.read_csv(next((root / 'pa').glob("xgb_f1*.csv")), index_col = 0)
rf_f1 = pd.read_csv(next((root / 'pa').glob("rf_f1*.csv")), index_col = 0)
rftv_f1 = pd.read_csv(next((root / 'pa').glob("rftv_f1*.csv")), index_col = 0)

cp = '26_hum_env_dhw_bathy_common_pa-2025-11-10_16-31'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

cp = '37_mm4_transductif-2025-12-02_12-47'
td_f1 = pd.read_csv(next((output_path / cp).glob("testF1*.csv")), index_col = 0)

In [66]:
# Bins

NBINS = 10

xgb_f1 = pd.read_csv(next((root / 'bins').glob(f"xgb{NBINS}_r2*.csv")), index_col = 0)
rf_f1 = pd.read_csv(next((root / 'bins').glob(f"rf{NBINS}_r2*.csv")), index_col = 0)
rftv_f1 = pd.read_csv(next((root / 'bins').glob(f"rftv{NBINS}_r2*.csv")), index_col = 0)

cp = '39_mm4_transductif_bins-2026-01-05_15-21'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '39_mm4_transductif_bins-2025-12-18_16-50'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

In [80]:
# Reg

cp = '38_mm4_transductif_reg-2025-12-17_18-53'
td_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

cp = '38_mm4_transductif_reg-2026-01-07_16-44'
mm_f1 = pd.read_csv(next((output_path / cp).glob("testR2--*.csv")), index_col = 0)

rfval_f1 = pd.read_csv(next((root / 'reg').glob("rftv_r2--*.csv")), index_col = 0)

In [4]:
# Load traits

traits = pd.read_csv('/data/data/RLS/traits.csv', index_col=0)

iucn_stats = {s: traits['IUCN_category'].get(s, 'NA') for s in td_f1.index}
iucn_encr = pd.Series([(iucn_stats[s] in ['CR', 'EN'])*1 for s in td_f1.index], index = td_f1.index)

trop_level = {s: traits['Troph'].get(s, 'NA') for s in td_f1.index}
trop_troph = pd.Series([trop_level[s] if trop_level[s] != 'NA' else None for s in td_f1.index], index = td_f1.index)
iucnL_stats = {s: traits['IUCN_inferred_Loiseau23'].get(s, 'NA') for s in td_f1.index}
iucnL_encr = pd.Series([(iucnL_stats[s] == 'Threatened')*1 for s in td_f1.index], index = td_f1.index)

In [24]:
# Calculate progress

scores = pd.concat([rf_f1["f1"], td_f1["f1"]], axis = 1)
names = ['Inductive', 'Transductive']
scores.columns = names
scores['progress'] = (scores[names[1]] - scores[names[0]])
scores['max'] = scores[names].max(axis=1)
best_scores = scores.loc[scores['max'] > 0.4]

In [25]:
# Associate to trait
traitname = 'IUCN_inferred_Loiseau23' # 'DemersPelag' 'Troph' 'IUCN_inferred_Loiseau23'
dic = {}

for s in best_scores.index:
    best_scores.loc[s, traitname] = traits[traitname].get(s, np.nan)

/tmp/ipykernel_198396/1268247608.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [26]:
import plotly.express as px


fig = px.violin(best_scores, x=traitname, y="progress", points = 'all', color = traitname, color_discrete_sequence= px.colors.qualitative.Pastel)


fig.update_layout(
    template='simple_white',
    width = 800, height=800,
    yaxis_title='Pearson delta',
    xaxis_title='',
    font=dict(size=20),
    showlegend=False)
    

fig.add_shape(type="line",
    xref="paper",
    x0=0, y0=0,
    x1=1, y1=0,
    line=dict(
        color="blue",
        width=3,
    ))

fig.show()

## RF old code

In [ ]:
# PA
 
for sp in tqdm(species):

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa')
    regr = RandomForestClassifier(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

    # Export predictions
    if regr.n_classes_ == 2:
        pd.Series(regr.predict_proba(X_train)[:, 1], name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_val)[:, 1], name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict_proba(X_test)[:, 1], name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")
    else:
        pd.Series([0] * len(Y_train), name=sp, index = Y_train.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"train_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_val), name=sp, index = Y_val.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"val_{sp.replace('/','-')}.csv")
        pd.Series([0] * len(Y_test), name=sp, index = Y_test.index).to_csv(root / 'pa' / 'rf-preds-tv' / f"test_{sp.replace('/','-')}.csv")

In [ ]:
# Bins 

for i in [5, 10, 20]:
    for sp in tqdm(species):

        X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa', i)
        regr = RandomForestClassifier(min_samples_split=2, # Useless
                                        min_samples_leaf=3,
                                        max_leaf_nodes=None,
                                        n_estimators=300,
                                        max_depth=None,
                                        random_state=0,
                                        n_jobs=32)

        # Export predictions

        regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

        # Export predictions
        pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"rf{i}-preds-tv" / f"test_{sp.replace('/','-')}.csv")


for i in [5, 10, 20]:
    for sp in tqdm(species):

        X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'pa', i)
        regr = RandomForestClassifier(min_samples_split=2, # Useless
                                        min_samples_leaf=3,
                                        max_leaf_nodes=None,
                                        n_estimators=300,
                                        max_depth=None,
                                        random_state=0,
                                        n_jobs=32)

        # Export predictions

        regr.fit(X_train, Y_train)

        # Export predictions
        pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"train_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"val_{sp.replace('/','-')}.csv")
        pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'bins' / f"rf{i}-preds" / f"test_{sp.replace('/','-')}.csv")

In [ ]:
# Reg

for sp in tqdm(species):

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'reg')
    regr = RandomForestRegressor(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(X_train, Y_train)

    # Export predictions
    pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'reg' / f"rf-preds" / f"train_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'reg' / f"rf-preds" / f"val_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'reg' / f"rf-preds" / f"test_{sp.replace('/','-')}.csv")

    X_train, Y_train, X_val, Y_val, X_test, Y_test = get_datasets(sp, 'rf', 'reg')
    regr = RandomForestRegressor(min_samples_split=2, # Useless
                                    min_samples_leaf=3,
                                    max_leaf_nodes=None,
                                    n_estimators=300,
                                    max_depth=None,
                                    random_state=0,
                                    n_jobs=32)

    # Export predictions

    regr.fit(pd.concat([X_train, X_val]), pd.concat([Y_train, Y_val]))

    # Export predictions
    pd.Series(regr.predict(X_train), name=sp, index = Y_train.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"train_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_val), name=sp, index = Y_val.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"val_{sp.replace('/','-')}.csv")
    pd.Series(regr.predict(X_test), name=sp, index = Y_test.index).to_csv(root / 'reg' / f"rf-preds-tv" / f"test_{sp.replace('/','-')}.csv")

## Other plots

#### 1v1 BlandAltman

In [ ]:
import pyCompare

pyCompare.blandAltman(scores[names[3]].values,scores[names[0]].values)

#### 1v1 Scatter plot

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

x, y = scores[names[0]].values, scores[names[3]].values

# Draw a combo histogram and scatterplot with density contours
f, ax = plt.subplots(figsize=(9, 9))
sns.scatterplot(x=x, y=y, s=5, color=".15")
sns.histplot(x=x, y=y, bins=40, pthresh=.1, cmap="mako")
sns.kdeplot(x=x, y=y, levels=5, color="w", linewidths=1)
sns.lineplot(x=[min(x), max(x)], y=[min(x), max(x)],color='red')
ax.set(xlabel=names[0] + ' F1', ylabel=names[3] + ' F1')

#### 1v1 Arrow Scatter

In [ ]:
scores['av'] = np.maximum(scores[names[0]], scores[names[3]])

best_scores = scores[scores['av'] > 0.3]
(best_scores[names[3]] - best_scores[names[2]]).hist(bins=40)

best_scores = scores.sort_values(by='XGBoost', ascending=False)
best_scores = best_scores[best_scores['av'] > 0.4]
best_scores['x'] = np.arange(len(best_scores))

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Example data
x = best_scores['x']
y1 = best_scores['XGBoost']
y2 = best_scores['Transductif']
hover_text = best_scores.index.to_list()

# Create the figure
fig = go.Figure()

# Add the two series as scatter plots
fig.add_trace(go.Scatter(
    x=x, y=y1,
    mode='markers',
    marker=dict(color=px.colors.qualitative.Pastel[0], size=10),
    name='XGBoost',
    hovertext=hover_text,
    hoverinfo='text'
))

fig.add_trace(go.Scatter(
    x=x, y=y2,
    mode='markers',
    marker=dict(color=px.colors.qualitative.Pastel[3], size=10),
    name='Transductive',
    hovertext=hover_text,
    hoverinfo='text'
))

# Add arrows for the differences
for i in range(len(x)):
    fig.add_annotation(
        ax=x[i], ay=y1[i],
        axref='x', ayref='y',
        x=x[i], y=y2[i],
        xref='x', yref='y',
        showarrow=True,
        arrowhead=1,
        arrowsize=1.5,
        arrowcolor=px.colors.qualitative.Pastel1[1],
        arrowwidth=1.5,
    )


fig.update_layout(
    template='simple_white',
    width = 1200, height=800,
    xaxis_title='Species rank',
    yaxis_title='F1 score',
    font=dict(size=20),
    legend=dict(yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99)
)
